# Grid Ablation on a Rented GPU

Does the bar grid the chorus model reads change where it places chorus starts?

Two models are trained on the same 231 songs with the same 49 validation songs.
They differ in one thing only: how their bar lines were drawn. **No Spotify
metadata is used anywhere in this notebook**: every tempo is measured from the
audio, and 4/4 is assumed, in training data and inference alike.

| Grid | How bars are drawn |
|---|---|
| librosa | Your own grid: take one BPM and one anchor beat, extrapolate bar lines forward and backward at a constant spacing. librosa only supplies the beat detection and a fallback tempo. |
| beat_this | Bars are taken from downbeats tracked by the Beat This! model |

The trials, in the order this notebook runs them:

| Trial | What it measures |
|---|---|
| T1 | Whether beat_this downbeats sit closer to the labelled boundaries than librosa grid lines. No model involved. |
| T2a | A model retrained on the librosa grid. This is the honest baseline. |
| T2b | A model retrained on the beat_this grid. T2b against T2a is the grid effect. |
| T0 | The released checkpoint, for reference only. Its training split may overlap this test set, so it is not a fair comparison. |
| T3a, T3b | Viterbi decoding instead of threshold-and-smooth, on both models. |
| T4a, T4b | Downbeat snapping added on top of Viterbi decoding, on both models. |
| T6a-T6c | The BPM-extrapolated grid with the tempo from three other estimators: Beat This! beat gaps, the full Mixxx "Queen Mary" pipeline, and Beat This! + Mixxx post-processing. Finds the estimator that builds the best grid. |

**Before you start, read this.** The code and the 331 audio files are staged in a
private Google Cloud Storage bucket. Step 2 asks you to sign in to Google once,
in a terminal on this instance, and everything after that downloads itself. If
you would rather not sign in, Step 5 shows how to copy the audio across with
`scp` instead.

The audio is commercial music held in a bucket that blocks public access. Keep it
that way.

Expected run time on one RTX 4090: roughly 2 to 4 hours, most of it building the
training data twice and training twice.

## Step 1 — Check the machine

Confirm a GPU is visible and that PyTorch can use it. If `cuda available` prints
`False`, stop: everything after this will run on the CPU and take many hours.

In [ ]:
!nvidia-smi

import torch
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Step 2 — Sign in to Google Cloud

The code and the audio are staged in a **private** Cloud Storage bucket, so this
machine has to prove who it is before it can download them.

Do this in a **terminal on this instance**, not in a notebook cell, because the
sign-in asks you a question and notebook cells cannot answer:

```
gcloud auth login --no-launch-browser
```

It prints a link. Open the link on your own computer, sign in as the account that
owns the bucket, and paste the code it gives you back into the terminal.

The cell below installs the Google Cloud CLI if the image lacks it, then checks
whether you are signed in. It stops if you are not.

**If you would rather not sign in at all**, read the `scp` note in Step 5 and copy
the files across from your own computer instead.

In [ ]:
import os, shutil, subprocess

BUCKET = "gs://dennis-computer-chorus-xfer"
REPO = "/workspace/chorus-detection"

if shutil.which("gcloud") is None:
    print("Installing the Google Cloud CLI, this takes a minute...")
    !curl -sSL https://sdk.cloud.google.com | bash -s -- --disable-prompts --install-dir=/usr/local > /dev/null 2>&1
    os.environ["PATH"] = "/usr/local/google-cloud-sdk/bin:" + os.environ["PATH"]

print("gcloud:", shutil.which("gcloud") or "NOT FOUND")

accounts = subprocess.run(["gcloud", "auth", "list", "--filter=status:ACTIVE",
                           "--format=value(account)"],
                          capture_output=True, text=True).stdout.strip()
print("signed in as:", accounts or "(nobody)")
assert accounts, (
    "Not signed in. Open a terminal on this instance and run:\n"
    "    gcloud auth login --no-launch-browser\n"
    "then run this cell again.")

## Step 3 — Download the code

This pulls a snapshot of the chorus-detection code taken from the branch that
holds the downbeat tracking, the Viterbi decoder, and the GPU matrix
factorisation. It is a snapshot rather than a clone, so it does not matter
whether that branch has been pushed to GitHub.

In [ ]:
!gcloud storage cp {BUCKET}/chorus-detection-src.tar.gz /workspace/
!tar -xzf /workspace/chorus-detection-src.tar.gz -C /workspace/

os.chdir(REPO)
print("working directory:", os.getcwd())

needed = ["pytorch_core/downbeats.py", "pytorch_core/decoding.py",
          "pytorch_core/evaluation.py", "pytorch_core/audio_processor.py",
          "scripts/preprocess.py", "data/clean_labeled.csv"]
missing = [f for f in needed if not os.path.exists(os.path.join(REPO, f))]
for f in needed:
    print(("  OK      " if f not in missing else "  MISSING ") + f)
assert not missing, f"the code snapshot is incomplete: {missing}"

## Step 4 — Install what is missing

Vast.ai images normally ship PyTorch built against the right CUDA version, so this
does **not** install or upgrade PyTorch. It installs the audio and science
packages, the Beat This! downbeat tracker, and the ffmpeg binary that the audio
library shells out to.

The last cell re-checks that PyTorch still sees the GPU, because a careless
dependency install can replace a CUDA build with a CPU one.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
!ffmpeg -version | head -1

!pip install -q "numpy>=1.24,<1.25" "scipy>=1.10,<1.11" "scikit-learn>=1.3,<1.4" "pandas>=2.0,<2.1" "librosa>=0.10,<0.11" "soundfile>=0.12,<0.13" "pydub>=0.25,<0.26" pyyaml tqdm matplotlib
!pip install -q "https://github.com/CPJKU/beat_this/archive/main.zip"

import importlib, torch
importlib.reload(torch)
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "PyTorch lost the GPU. Reinstall a CUDA build before continuing."

## Step 5 — Download the audio

The 331 mp3 files are about 1 GB, so this is the slowest download. They land in
`/workspace/chorus-detection/data/audio/processed/`, named by song ID.

One song (ID 20) has a label but no audio file, so **331 files is a complete
set**, not a shortfall.

**Copyright note.** These are commercial recordings. The bucket blocks public
access on purpose. Do not republish this archive or move it to a public location.

**If you skipped the Google sign-in**, run these on your own computer instead,
using the SSH host and port from your instance page:

```
tar -cf audio.tar -C "data/audio" processed
scp -P <PORT> audio.tar root@<HOST>:/workspace/
```

In [ ]:
import glob

AUDIO_DIR = os.path.join(REPO, "data", "audio", "processed")
os.makedirs(os.path.join(REPO, "data", "audio"), exist_ok=True)

if not os.path.exists("/workspace/audio-processed.tar"):
    !gcloud storage cp {BUCKET}/audio-processed.tar /workspace/
!tar -xf /workspace/audio-processed.tar -C {REPO}/data/audio/

found = sorted(glob.glob(os.path.join(AUDIO_DIR, "*.mp3")))
print(f"audio files found: {len(found)}")
assert len(found) > 0, (
    "No audio found in " + AUDIO_DIR + ". Either the download failed or the "
    "archive extracted somewhere else.")
if len(found) < 331:
    print(f"WARNING: expected 331 files, found {len(found)}. Trials will run on "
          "fewer songs and the numbers will not compare to a full run.")
else:
    print("complete set")

## Step 6 — Write the frozen split

The train, validation and test song lists are fixed here rather than regenerated,
so this run is comparable with earlier ones. The test songs are never trained on.

In [ ]:
import os

SPLITS_DIR = os.path.join(REPO, "data", "splits")
os.makedirs(SPLITS_DIR, exist_ok=True)

TRAIN_IDS = "139 78 207 40 18 350 527 367 423 401 209 108 79 292 512 299 275 311 21 41 266 302 548 405 104 267 498 86 348 83 352 56 19 229 453 374 37 54 165 282 149 161 450 368 273 206 363 164 257 70 249 451 415 12 265 472 106 178 313 445 550 546 493 102 212 464 183 441 214 501 127 138 476 89 508 362 223 268 308 462 280 305 360 240 71 76 145 452 371 55 457 95 310 182 283 118 105 509 243 144 132 530 75 179 438 109 544 448 22 326 158 479 26 328 73 331 290 403 197 422 129 400 459 304 13 353 242 215 20 424 250 252 524 2 135 156 468 199 507 309 291 358 234 528 324 529 314 478 407 439 33 523 296 157 489 52 349 269 461 440 281 67 82 58 372 426 327 447 101 279 357 51 287 490 540 293 208 143 227 72 264 370 92 3 545 163 66 196 463 23 375 201 334 114 153 142 355 307 456 298 413 172 25 7 366 416 29 160 239 526 420 301 140 90 306 553 1 552 534 6 256 399 535 437 43 259 356 254 154 481 185 32".split()
VAL_IDS = "166 470 520 11 93 410 551 141 499 454 458 103 289 147 474 347 487 533 466 303 186 433 300 69 294 107 195 116 329 159 65 514 434 543 460 421 428 61 48 88 64 510 412 341 162 436 393 494 260".split()
TEST_IDS = "176 10 170 219 418 151 502 241 175 504 465 115 236 517 351 91 30 8 467 177 171 36 389 98 38 354 180 168 46 480 131 336 522 397 547 491 68 317 297 321 255 235 430 288 99 130 388 203 261 511 258".split()

for name, ids in [("train_songs.txt", TRAIN_IDS),
                  ("val_songs.txt", VAL_IDS),
                  ("test_songs.txt", TEST_IDS)]:
    with open(os.path.join(SPLITS_DIR, name), "w") as handle:
        handle.write("\n".join(ids) + "\n")

print(f"train {len(TRAIN_IDS)}  validation {len(VAL_IDS)}  test {len(TEST_IDS)}")
assert not (set(TRAIN_IDS) & set(TEST_IDS)), "a training song leaked into the test set"
assert not (set(VAL_IDS) & set(TEST_IDS)), "a validation song leaked into the test set"
print("no song appears in more than one split")

## Step 7 — Shared setup

Loads the configuration and the labels, and defines the helpers every trial uses:
turning a model's per-bar output into chorus segments, and scoring those segments
against the labelled truth.

In [ ]:
import sys, warnings
import numpy as np, pandas as pd, librosa
warnings.filterwarnings("ignore")
sys.path.insert(0, REPO)

from pytorch_core.audio_processor import process_audio
from pytorch_core.model import smooth_predictions, load_CRNN_model
from pytorch_core.decoding import viterbi_chorus
from pytorch_core.downbeats import track_downbeats, snap_chorus_segments, is_available
from pytorch_core import evaluation as ev
from scripts.inference import load_config, load_model

CONFIG = load_config(os.path.join(REPO, "config", "default.yaml"))
LABELS = pd.read_csv(os.path.join(REPO, "data", "clean_labeled.csv"))
RESULTS_CSV = os.path.join(REPO, "results", "trials.csv")
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
print("beat_this installed:", is_available())


def song_rows(song_id):
    return LABELS[LABELS["SongID"].astype(str) == str(song_id)]


def true_segments(song_id):
    rows = song_rows(song_id)
    rows = rows[rows["label"] == "chorus"].sort_values("start_time")
    starts, ends = [], []
    for _, r in rows.iterrows():
        if ends and abs(r["start_time"] - ends[-1]) < 1e-6:
            ends[-1] = float(r["end_time"])
        else:
            starts.append(float(r["start_time"])); ends.append(float(r["end_time"]))
    return starts, ends


# Tempo estimators for the BPM-extrapolated grid. No Spotify metadata is used
# anywhere: every tempo below is measured from the audio itself. Each estimator
# is cached per song, because some are slow and several trials share them.
_tempo_cache = {}


def tempo_librosa(path):
    """Baseline: librosa onset-strength beat tracking (Ellis 2007)."""
    y, sr = librosa.load(path, sr=22050)
    t, _ = librosa.beat.beat_track(y=y, sr=sr)
    return float(np.atleast_1d(t)[0])


def tempo_beat_this(path):
    """Median inter-beat interval of Beat This! beats (ISMIR 2024)."""
    beats, _ = track_downbeats(path, device=DEVICE)
    gaps = np.diff(np.sort(np.asarray(beats, float)))
    gaps = gaps[gaps > 0]
    return float(60.0 / np.median(gaps)) if gaps.size else None


def tempo_qm(path):
    """The Mixxx "Queen Mary" method, end to end: Complex Spectral Difference
    onsets and Viterbi beat tracking (ported in scripts/script.py), then
    Mixxx's constant-region search and BPM rounding (pytorch_core/mixxx_bpm)."""
    from scripts.script import BeatAnalyzer
    from pytorch_core.mixxx_bpm import calculate_bpm
    global _qm_analyzer
    if "_qm_analyzer" not in globals():
        _qm_analyzer = BeatAnalyzer()
    result = _qm_analyzer.analyze(path)
    return calculate_bpm(result.beats_seconds)


def tempo_beat_this_mixxx(path):
    """Beat This! beats fed through Mixxx's constant-region BPM. Combines the
    most accurate tracker with the post-processing that makes Mixxx's BPM
    stable and rounded."""
    from pytorch_core.mixxx_bpm import calculate_bpm
    beats, _ = track_downbeats(path, device=DEVICE)
    return calculate_bpm(np.asarray(beats, dtype=float))


TEMPO_ESTIMATORS = {"librosa": tempo_librosa, "beat_this": tempo_beat_this,
                    "qm": tempo_qm, "beat_this_mixxx": tempo_beat_this_mixxx}


def estimate_tempo(song_id, method):
    key = (song_id, method)
    if key not in _tempo_cache:
        _tempo_cache[key] = TEMPO_ESTIMATORS[method](audio_path(song_id))
    return _tempo_cache[key]


def audio_path(song_id):
    return os.path.join(AUDIO_DIR, f"{song_id}.mp3")


def segments_from_binary(binary, grid_times):
    """Group runs of chorus bars into (start_times, end_times)."""
    starts, ends = [], []
    idx = np.where(np.asarray(binary) == 1)[0]
    if idx.size == 0:
        return starts, ends
    grp = [idx[0]]
    for i in idx[1:]:
        if i == grp[-1] + 1:
            grp.append(i)
        else:
            starts.append(float(grid_times[grp[0]])); ends.append(float(grid_times[grp[-1] + 1])); grp = [i]
    starts.append(float(grid_times[grp[0]])); ends.append(float(grid_times[grp[-1] + 1]))
    return starts, ends

In [ ]:
def predict(model, song_id, grid_source, decode="smooth", snap=False,
            tempo_source="librosa"):
    """Run one song through a model and return its chorus segments.

    tempo_source names the estimator feeding the BPM-extrapolated grid
    (see TEMPO_ESTIMATORS). It is ignored by the beat_this grid, which needs
    no global tempo. Nothing here reads outside metadata.
    """
    path = audio_path(song_id)
    if not os.path.exists(path):
        return None
    ts = None    # beats per bar: 4/4 assumed, matching the training data build
    bpm = None
    if grid_source == "librosa" and tempo_source != "librosa":
        # librosa is what process_audio measures on its own; other estimators
        # are computed here and passed in.
        bpm = estimate_tempo(song_id, tempo_source)
    # The audio is already silence-stripped and the labels were made on it,
    # so it must not be trimmed again here.
    processed, af = process_audio(path, trim_silence=False,
                                  sr=CONFIG["data"]["sr"],
                                  hop_length=CONFIG["data"]["hop_length"],
                                  bpm=bpm, time_signature=ts,
                                  grid_source=grid_source, device=DEVICE)
    if processed is None:
        return None
    with torch.no_grad():
        out = model(torch.tensor(processed, dtype=torch.float32).to(DEVICE)).cpu().numpy().squeeze()
    n_meters = min(len(af.meter_grid) - 1, len(out))
    probs = out[:n_meters]
    binary = viterbi_chorus(probs, switch_penalty=2.0, min_bars=4) if decode == "viterbi" \
        else smooth_predictions(probs)
    grid_times = librosa.frames_to_time(af.meter_grid, sr=af.sr, hop_length=af.hop_length)
    starts, ends = segments_from_binary(binary, grid_times)
    beat_period = 60.0 / af.tempo if af.tempo else None
    if snap and starts and is_available():
        _, downbeats = track_downbeats(path, device=DEVICE)
        if len(downbeats) >= 2:
            rms = np.asarray(af.rms).ravel()
            rms_t = librosa.frames_to_time(np.arange(rms.size), sr=af.sr, hop_length=af.hop_length)
            starts, ends = snap_chorus_segments(starts, ends, downbeats,
                                                energy=rms, energy_times=rms_t)
    duration = len(af.y) / af.sr
    return starts, ends, beat_period, duration


# Every scored song is kept, not just the trial averages, so any graph in this
# notebook can be rebuilt later from the saved file without re-running anything.
SONG_ROWS = []

# A predicted chorus start within this fraction of a bar counts as landing on it.
EXACT_FRACTION = 0.25


def anchor_accuracy(pred_starts, true_starts, beat_period, beats_per_bar=4):
    """How far the FIRST chorus start is from the labelled one.

    This is the number the mixing project needs. The first chorus start is the
    point an incoming track is aligned to, so an error here moves the whole
    transition, however well the later choruses were found.
    """
    if not true_starts or not beat_period:
        return None, None
    bar = beat_period * beats_per_bar
    if not pred_starts:
        return None, "off"
    error = pred_starts[0] - true_starts[0]
    size = abs(error)
    verdict = ("exact" if size <= EXACT_FRACTION * bar
               else "within_1_bar" if size <= bar else "off")
    return error, verdict


def run_trial(trial_id, description, model, grid_source, decode="smooth", snap=False,
              tempo_source="dataset", test_ids=None):
    """Score one configuration over the test songs. Returns its summary row."""
    rows = []
    for sid in (test_ids or TEST_IDS):
        r = predict(model, sid, grid_source, decode=decode, snap=snap,
                    tempo_source=tempo_source)
        if r is None:
            continue
        ps, pe, bp, dur = r
        ts_, te_ = true_segments(sid)
        scored = ev.score_song(ps, pe, ts_, te_, duration=dur, beat_period=bp)
        rows.append(scored)

        error, verdict = anchor_accuracy(ps, ts_, bp)
        SONG_ROWS.append({"trial": trial_id, "song_id": sid, "grid": grid_source,
                          "decode": decode, "snap": int(snap),
                          "tempo_source": tempo_source,
                          "beat_period_s": bp,
                          "true_first_chorus_s": ts_[0] if ts_ else None,
                          "pred_first_chorus_s": ps[0] if ps else None,
                          "anchor_error_s": error,
                          "anchor_error_bars": (error / (bp * 4)) if (error is not None and bp) else None,
                          "anchor_verdict": verdict, **scored})

    agg = ev.aggregate(rows)
    verdicts = [r["anchor_verdict"] for r in SONG_ROWS
                if r["trial"] == trial_id and r["anchor_verdict"]]
    n = len(verdicts) or 1
    exact = verdicts.count("exact")
    within = exact + verdicts.count("within_1_bar")

    record = {"trial": trial_id, "description": description, "grid": grid_source,
              "decode": decode, "snap": int(snap),
              "tempo_source": tempo_source, "n_songs": len(rows),
              "anchor_exact_rate": exact / n,
              "anchor_within_1_bar_rate": within / n, **agg}
    print(f"[{trial_id}] {description}  (n={len(rows)})")
    print(f"    first chorus start: {exact/n:.1%} exact, {within/n:.1%} within one bar")
    for k in ("f1_mean", "median_abs_err_s_median", "median_abs_err_beats_median"):
        if k in agg:
            print(f"    {k}: {agg[k]:.3f}")
    return record


TRIAL_ROWS = []

## T1 — Are beat_this downbeats closer to the labelled boundaries?

No model runs here. For every labelled chorus boundary in the test songs, this
measures the distance to the nearest librosa grid line and to the nearest
beat_this downbeat, in beats.

If the librosa distances cluster near one beat while the beat_this distances sit
near zero, the librosa grid is out of phase, and that alone would push every
predicted boundary off. Takes a few minutes.

In [ ]:
def grid_line_times(song_id, grid_source):
    path = audio_path(song_id)
    bpm, ts = song_meta(song_id)
    _, af = process_audio(path, trim_silence=False, sr=CONFIG["data"]["sr"],
                          hop_length=CONFIG["data"]["hop_length"], bpm=bpm,
                          time_signature=ts, grid_source=grid_source, device=DEVICE)
    beat_period = 60.0 / af.tempo if af.tempo else None
    return librosa.frames_to_time(af.meter_grid, sr=af.sr, hop_length=af.hop_length), beat_period


librosa_err, beatthis_err = [], []
for sid in TEST_IDS:
    if not os.path.exists(audio_path(sid)):
        continue
    ts_, te_ = true_segments(sid)
    truth = np.array(ts_ + te_)
    if truth.size == 0:
        continue
    lt, bp = grid_line_times(sid, "librosa")
    if bp:
        librosa_err += list(np.min(np.abs(lt[None, :] - truth[:, None]), axis=1) / bp)
    if is_available():
        _, downbeats = track_downbeats(audio_path(sid), device=DEVICE)
        if len(downbeats) and bp:
            db = np.asarray(downbeats)
            beatthis_err += list(np.min(np.abs(db[None, :] - truth[:, None]), axis=1) / bp)

print(f"boundaries measured: librosa {len(librosa_err)}, beat_this {len(beatthis_err)}")
for name, errs in [("librosa grid", librosa_err), ("beat_this downbeats", beatthis_err)]:
    if errs:
        e = np.array(errs)
        print(f"{name:22s} median {np.median(e):.3f} beats | "
              f"within 0.25 beat: {100*np.mean(e <= 0.25):.1f}% | "
              f"within 1 beat: {100*np.mean(e <= 1.0):.1f}%")

In [ ]:
import matplotlib.pyplot as plt

if librosa_err and beatthis_err:
    plt.figure(figsize=(9, 4))
    bins = np.linspace(0, 2, 41)
    plt.hist(librosa_err, bins=bins, alpha=0.6, label="librosa grid")
    plt.hist(beatthis_err, bins=bins, alpha=0.6, label="beat_this downbeats")
    plt.xlabel("distance from labelled boundary to nearest bar line (beats)")
    plt.ylabel("number of boundaries")
    plt.title("T1 - which grid sits closer to the labelled chorus boundaries")
    plt.legend(); plt.tight_layout(); plt.show()

## Step 8 — Build the training data, once per grid

Each song is turned into per-bar features and per-bar labels. This runs twice,
writing to separate directories, because the two grids cut the song into
different bars.

This is the slowest step. One song failing with `audio file not found` is
expected: song 20 has a label but no audio.

In [ ]:
import subprocess

def has_flag(flag):
    """True when the preprocessing script accepts this option."""
    out = subprocess.run([sys.executable, "scripts/preprocess.py", "--help"],
                         cwd=REPO, capture_output=True, text=True)
    return flag in out.stdout


NMF_FLAG = ["--nmf-device", DEVICE] if has_flag("--nmf-device") else []
if not NMF_FLAG:
    print("This clone has no --nmf-device option, so matrix factorisation runs on the CPU.")
    print("Push the newer commit from your local machine to make this step faster.")


def build_training_data(grid_source, segments_dir, labels_dir):
    command = [sys.executable, "scripts/preprocess.py",
               "--grid-source", grid_source,
               "--segments-dir", segments_dir,
               "--labels-dir", labels_dir,
               "--tempo-source", "estimated",
               "--device", DEVICE] + NMF_FLAG
    subprocess.run(command, cwd=REPO)   # a missing song must not stop the run
    made = len(glob.glob(os.path.join(REPO, segments_dir, "*_data.pkl")))
    print(f"{grid_source} grid: {made} songs ready")
    assert made > 0, f"no training data was produced for the {grid_source} grid"
    return made


build_training_data("librosa", "data/seg_librosa", "data/lab_librosa")
build_training_data("beat_this", "data/seg_beatthis", "data/lab_beatthis")

## T2 — Train one model per grid

Same songs, same split, same seed. Only the grid differs, so any difference
between these two models is caused by the grid.

In [ ]:
from torch.utils.data import DataLoader
from pytorch_core.data.dataset import ChorusDataset
from pytorch_core.models.crnn import CRNN
from pytorch_core.training.trainer import Trainer


def available_ids(ids, segments_dir):
    return [s for s in ids
            if os.path.exists(os.path.join(REPO, segments_dir, f"{s}_data.pkl"))]


def train_model(segments_dir, labels_dir, checkpoint_dir):
    seg, lab = os.path.join(REPO, segments_dir), os.path.join(REPO, labels_dir)
    train_ids, val_ids = available_ids(TRAIN_IDS, segments_dir), available_ids(VAL_IDS, segments_dir)
    print(f"training on {len(train_ids)} songs, validating on {len(val_ids)}")

    batch = CONFIG["training"]["batch_size"]
    train_loader = DataLoader(ChorusDataset(train_ids, seg, lab, CONFIG), batch_size=batch, shuffle=True)
    val_loader = DataLoader(ChorusDataset(val_ids, seg, lab, CONFIG), batch_size=batch, shuffle=False)

    torch.manual_seed(42); np.random.seed(42)
    trainer = Trainer(CRNN(CONFIG), CONFIG, train_loader, val_loader,
                      checkpoint_dir=os.path.join(REPO, checkpoint_dir), device=DEVICE)
    trainer.train(epochs=CONFIG["training"]["epochs"])
    best = os.path.join(REPO, checkpoint_dir, "best_model.pt")
    return load_model(best, CONFIG).to(DEVICE).eval()


model_librosa = train_model("data/seg_librosa", "data/lab_librosa", "models/ablation_librosa")

In [ ]:
model_beatthis = train_model("data/seg_beatthis", "data/lab_beatthis", "models/ablation_beatthis")

## Score every trial

T0 first, for reference. Read it with care: the released checkpoint was trained on
a different, older split that may include songs in this test set, so it can look
better than it deserves. T2a is the baseline to trust.

In [ ]:
pretrained = load_CRNN_model(os.path.join(REPO, "models", "CRNN_pytorch", "crnn_v1.pt"))
pretrained.to(DEVICE).eval()
TRIAL_ROWS.append(run_trial("T0", "released checkpoint, librosa grid, smooth", pretrained, "librosa"))

In [ ]:
TRIAL_ROWS.append(run_trial("T2a", "retrained librosa grid, smooth", model_librosa, "librosa"))
TRIAL_ROWS.append(run_trial("T2b", "retrained beat_this grid, smooth", model_beatthis, "beat_this"))

In [ ]:
TRIAL_ROWS.append(run_trial("T3a", "librosa grid, viterbi", model_librosa, "librosa", decode="viterbi"))
TRIAL_ROWS.append(run_trial("T3b", "beat_this grid, viterbi", model_beatthis, "beat_this", decode="viterbi"))

In [ ]:
TRIAL_ROWS.append(run_trial("T4a", "librosa grid, viterbi + snap", model_librosa, "librosa", decode="viterbi", snap=True))
TRIAL_ROWS.append(run_trial("T4b", "beat_this grid, viterbi + snap", model_beatthis, "beat_this", decode="viterbi", snap=True))

### T6 — which tempo estimator builds the best grid?

The BPM-extrapolated grid stands or falls on its one tempo value. These trials
re-run the same model and decoding while swapping only the tempo estimator:

| Method | What it is |
|---|---|
| librosa | Onset-strength beat tracking (Ellis 2007). The baseline; every trial above used it. |
| beat_this | Median gap between beats tracked by Beat This! (ISMIR 2024). |
| qm | The Mixxx "Queen Mary" method, end to end: spectral-difference onsets, Viterbi beat tracking, then Mixxx's constant-region search and BPM rounding. This is what Mixxx ships as its default analyzer. |
| beat_this_mixxx | Beat This! beats fed through the same Mixxx constant-region post-processing. The strongest tracker combined with the post-processing that makes Mixxx report "120.00" instead of "119.87". |

An estimator that fails on the probe song is skipped and reported, and costs
only its own trial.

In [ ]:
shootout = [("T6a", "beat_this"), ("T6b", "qm"), ("T6c", "beat_this_mixxx")]
for trial_id, method in shootout:
    try:
        estimate_tempo(TEST_IDS[0], method)   # fail fast before 51 songs
    except Exception as error:
        print(f"[{trial_id}] skipped: the {method} estimator failed ({error})")
        continue
    TRIAL_ROWS.append(run_trial(trial_id, f"librosa grid, {method} tempo, viterbi",
                                model_librosa, "librosa", decode="viterbi",
                                tempo_source=method))

## Results

Two files are written, and the second one is the important one for later:

| File | One row per | Why it exists |
|---|---|---|
| `results/trials.csv` | trial | The summary table below |
| `results/trial_songs.csv` | song, per trial | Every graph here can be rebuilt from this on your own machine |

How to read the table:

- **T2b against T2a** is the grid effect, the question this notebook exists to answer.
- **T3 against T2** is what Viterbi decoding adds.
- **T4 against T3** is what downbeat snapping adds.
- **T0 is not a fair comparison.** The released checkpoint was trained on an older
  split that may include songs in this test set.

`anchor_exact_rate` is the share of songs whose **first** chorus start landed on
the labelled one, within a quarter of a bar. That is the number the mixing project
needs, because the first chorus start is what an incoming track gets aligned to.
Lower `median_abs_err` is better; higher rates and `f1_mean` are better.

In [ ]:
trials = pd.DataFrame(TRIAL_ROWS)
songs = pd.DataFrame(SONG_ROWS)

trials.to_csv(RESULTS_CSV, index=False)
SONGS_CSV = os.path.join(REPO, "results", "trial_songs.csv")
songs.to_csv(SONGS_CSV, index=False)
print(f"wrote {RESULTS_CSV}  ({len(trials)} rows)")
print(f"wrote {SONGS_CSV}  ({len(songs)} rows)")

cols = [c for c in ["trial", "description", "grid", "decode", "snap",
                    "tempo_source", "n_songs",
                    "anchor_exact_rate", "anchor_within_1_bar_rate", "f1_mean",
                    "median_abs_err_s_median", "median_abs_err_beats_median"]
        if c in trials.columns]
trials[cols].round(3)

### Graphs

Four views of the same results. Each is drawn from the two tables above, so you
can reproduce any of them later from `trial_songs.csv`.

In [ ]:
import matplotlib.pyplot as plt

order = [t for t in ["T0", "T2a", "T2b", "T3a", "T3b", "T4a", "T4b",
                     "T6a", "T6b", "T6c"]
         if t in set(trials["trial"])]
tr = trials.set_index("trial").loc[order]
colour = ["#999999" if t == "T0" else "#2ca02c" if t.startswith("T6")
          else "#1f77b4" if t.endswith("a") else "#ff7f0e" for t in order]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. First chorus start placement, the number this project needs.
ax = axes[0][0]
x = np.arange(len(order))
ax.bar(x - 0.2, tr["anchor_exact_rate"] * 100, 0.4, label="exact", color=colour)
ax.bar(x + 0.2, tr["anchor_within_1_bar_rate"] * 100, 0.4, label="within 1 bar",
       color=colour, alpha=0.45)
ax.axhline(80, ls="--", c="grey", lw=1)
ax.axhline(95, ls=":", c="grey", lw=1)
ax.set_xticks(x); ax.set_xticklabels(order)
ax.set_ylabel("% of test songs"); ax.set_ylim(0, 100)
ax.set_title("First chorus start placement (dashed 80%, dotted 95% targets)")
ax.legend(fontsize=8)

# 2. Median boundary error in beats, so tempo does not distort the comparison.
ax = axes[0][1]
key = "median_abs_err_beats_median"
if key in tr.columns:
    ax.bar(x, tr[key], color=colour)
    ax.set_xticks(x); ax.set_xticklabels(order)
    ax.set_ylabel("beats"); ax.set_title("Median boundary error (lower is better)")

# 3. Frame F1, how much of each chorus is covered.
ax = axes[1][0]
ax.bar(x, tr["f1_mean"], color=colour)
ax.set_xticks(x); ax.set_xticklabels(order)
ax.set_ylabel("F1"); ax.set_ylim(0, 1)
ax.set_title("Frame F1 (higher is better)")

# 4. Spread of first-chorus error per song, not just the average.
ax = axes[1][1]
data = [songs.loc[songs["trial"] == t, "anchor_error_bars"].dropna().values for t in order]
if any(len(d) for d in data):
    ax.boxplot(data, labels=order, showfliers=True)
    ax.axhline(0, c="grey", lw=1)
    ax.set_ylabel("bars (predicted minus true)")
    ax.set_title("First chorus start error per song")

plt.tight_layout(); plt.show()

## Take the results home

The rented machine disappears when you stop it. Save both result files and the two
checkpoints before that happens.

Run this **on your own computer**, with the SSH host and port from your instance page:

```
scp -P <PORT> root@<HOST>:/workspace/chorus-detection/results/trials.csv .
scp -P <PORT> root@<HOST>:/workspace/chorus-detection/results/trial_songs.csv .
scp -P <PORT> root@<HOST>:/workspace/chorus-detection/models/ablation_librosa/best_model.pt ./librosa_grid_model.pt
scp -P <PORT> root@<HOST>:/workspace/chorus-detection/models/ablation_beatthis/best_model.pt ./beatthis_grid_model.pt
```

With `trial_songs.csv` on your own machine you can redraw every graph above, and
any other you want, without the GPU.

The cell below reports what there is to copy.

In [ ]:
for label, path in [("trial summary", RESULTS_CSV),
                    ("per-song results", os.path.join(REPO, "results/trial_songs.csv")),
                    ("librosa grid model", os.path.join(REPO, "models/ablation_librosa/best_model.pt")),
                    ("beat_this grid model", os.path.join(REPO, "models/ablation_beatthis/best_model.pt"))]:
    state = f"{os.path.getsize(path)/1e6:8.1f} MB" if os.path.exists(path) else " MISSING  "
    print(f"{label:22s} {state}  {path}")